# asyncio - await专题

## 一、await 的本质含义
在异步编程中，await 并不是一个简单的“等待”动作，而是一个“出让执行权”的信号。它告诉程序的调度员（事件循环）：我现在需要等一个结果，暂时用不到 CPU，你可以先把 CPU 拿走去处理清单上其他 **已经报了名** 的任务。等我等的东西好了，你再回来叫醒我，从这行代码继续往下跑。

### 1.1 关键辨析：哪些是“提交了任务”，哪些不是？
这是初学者最容易产生误解的地方，理解这一点就能区分“并发”和“串行”。
- 使用 await 后面接函数调用（例如：await func()）：
这是“立即提交并原地死等”。你把报案单交给了警察，但你搬了个小板凳坐在警察局门口，不处理完你的案子你就不走。虽然你让出了 CPU，但因为你没有提前提交后续的其他任务，CPU 往往只能闲着。
- 使用 asyncio.create_task(func())：
这是真正的“后台提交”。这行代码跑完的瞬间，任务就正式进入了事件循环的待办清单。即使你还没写 await，这个任务其实已经开始在后台计时或排队跑了。
- 使用 asyncio.gather(func1(), func2())：
这是“批量提交”。它瞬间把多个报案单一起塞进警察局，然后返回一个总的凭证。你再 await 这个总凭证，就能实现多个任务同时跑。
- **大坑：直接调用异步函数（例如：func()）：**
这不是提交任务。这只是创建了一个“协程对象”，它就像一张填好的报案单，但你还把它揣在兜里，没有交给警察局。此时代码不会运行。

### 1.2 await 的使用方法
- 必须在异步环境中使用：
await 关键字只能出现在被 async def 定义的函数（协程）内部。在普通函数里写 await 会直接报语法错误。
- 后面必须接“可等待对象”：
await 后面通常接三种东西：直接调用异步函数产生的协程对象、由 create_task 创建的任务对象、或者是 Ray 的 ObjectRef（对象引用）。如果你尝试 await 一个普通的整数或字符串，程序会崩溃。
- 链式调用的规则：
你可以 await 一个返回协程的函数，也可以 await 一个变量，只要这个变量指向的是上述的可等待对象。

### 1.3 核心注意事项（避坑指南）
- 防止“毒化”循环：
千万不要在 await 附近写 time.sleep()。await 是谦让 CPU，而 time.sleep 是霸占 CPU 睡觉。一旦霸占，整个事件循环就瘫痪了，所有后台任务都会跟着一起卡死。
- 避免 CPU 密集型长计算：
如果一行代码不带 await，且需要跑好几秒（比如一个几亿次的循环），它同样会卡死事件循环。这种活儿应该交给 Ray 的远程任务去做，而不是在 asyncio 里硬扛。
- 异常处理的滞后性：
如果你 create_task 提交了一个任务但没有 await 它，这个任务如果报错了，你的主程序可能完全不知道，直到程序结束才弹出一个隐晦的警告。建议重要的任务最后都要 await 一下，或者通过回调来捕捉错误。
- 串行陷阱：
如果你有多个互不相关的任务，千万不要写成一排 await。记住：先用 create_task 提交，或者用 gather 打包，最后再 await。

### 1.4 相关扩展与进阶内容
- 超时控制：
可以使用 asyncio.wait_for 来包裹一个协程，并设置一个时间上限。如果 await 的时间超过了限制，它会自动抛出超时异常并尝试取消掉那个任务，这是保护系统响应速度的重要手段。
- 现代化的任务组（TaskGroup）：
在 Python 3.11 之后，官方推荐使用 async with asyncio.TaskGroup() 这种写法。它比 gather 更安全，如果组内一个任务崩了，它会自动确保其他任务也被正确关闭，避免产生“孤儿任务”。
- 迭代器与上下文：
除了 await 函数，还有异步循环（async for）和异步上下文管理器（async with）。它们本质上是在进入循环或进入代码块时，内部执行了 await 动作。
- 与 Ray 的联动：
Ray 的设计非常精妙，它让分布在几百台机器上的任务返回的“取货码”也能被 await。当你 await 一个 Ray 的引用时，你其实是在单机异步的基础上，实现了跨机器的非阻塞等待。

总结一句话：先利用 create_task 填满你的“待办清单”，再通过 await 优雅地交出 CPU 权限，这才是高性能异步程序的正确打开方式。

## 二、Asyncio & Ray

### 2.1 核心哲学：谁是“勤快”的，谁是“懒惰”的？
在 Asyncio 中，直接调用异步函数是“极其懒惰”的。它只是生成了一个协程对象，如果你不手动通过 create_task 把它塞进循环，或者用 await 去激活它，它永远不会执行。

而在 Ray 中，调用 remote 函数是“极其勤快”的。当你执行“函数.remote()”的那一秒，任务就已经被提交给了 Ray 的调度器。哪怕你现在不去管它，Ray 也会在集群里找一台空闲的机器开始运行。它返回的 ObjectRef（对象引用）不是一段代码，而是一个已经“在路上”的任务凭证。

### 2.2 提交动作的对等关系
如果我们把两者的操作进行对等，你会发现：Ray 的“函数.remote()”其实在语义上等同于 Asyncio 的“create_task”。
它们都完成了“把活派出去”的动作，并立刻返回一个可以追踪进度的句柄（在 Ray 里叫 ObjectRef，在 Asyncio 里叫 Task）。

如果你在代码里写了一行“函数.remote()”却没写“ray.get”或“await”，这个任务依然会在远程跑完。这和 Asyncio 中写了“create_task”却没写“await”是一样的，任务会在后台默默完成。

### 2.3 关键桥梁：ObjectRef 也是可等待对象
Ray 的设计最天才的地方在于，它让分布式任务完美兼容了 Asyncio 的 await 语法。
当你身处一个异步函数中，你可以直接 await 一个 Ray 的引用。这意味着：
- 任务是在远程机器上跑的（物理并行）。
- 主程序的 CPU 并没有死等，而是利用 await 释放了控制权，去处理本地的其他协程（逻辑并发）。

这种“跨机器提交，本地异步等待”的组合，是高性能分布式系统的终极形态。

### 2.4 哪些操作发送了任务？
在 Ray + Asyncio 的混合架构中，任务提交的判断准则如下：
- 纯 Ray 提交：调用 remote 方法。这会跨越进程边界，将任务发送到 Ray 的任务队列中。这不需要 await 也会运行。
- 纯 Asyncio 提交：调用 create_task。这是在本地线程的清单里增加一个记录。这也不需要 await 也会运行。
- 混合等待：await 一个 remote 调用。这实际上是两个动作的合并：首先利用 Ray 把任务发到远端核心跑，然后利用 Asyncio 把主线程的等待时间让给其他任务。

### 2.5 注意事项：不要混淆“拿结果”的手段
在 Ray 中，拿结果有两种手段，它们的阻塞层级完全不同：
- ray.get(引用)：这是“硬索取”。它会阻塞整个进程。如果你在异步函数里用了它，你会把整个 asyncio 事件循环给“毒死”，导致本地所有的并发任务都卡住。
- await 引用：这是“软等待”。它只挂起当前的这一个函数，让 CPU 能够去处理本地的其他工作，直到远端的 Ray 任务返回结果。

### 2.6 架构建议：先群发，后汇总
在实际的 Ray 架构中，最推荐的模式是：
- 利用循环连续调用多次 .remote()，瞬间把成百上千个任务发往集群各个节点。这时，你手里握着一堆 ObjectRef（取货码）。
- 接着，使用 asyncio.gather 配合 await，一次性监听这所有的取货码。

这样做，你既利用了 Ray 的多机算力（群殴），又利用了 Asyncio 的单核切换效率（分身），实现了真正的全局资源利用最大化。

##  三、代码阅读

### 2.1 await 的本质
Q: 请观察以下代码，并预测其总执行时间和打印顺序。

A: 打印顺序是A start，A end, B start，B end，用时3秒

注意：await 会挂起当前的协程，直到被等待的目标完全运行结束。

- 在 main 函数里：
  - 执行到 res_a = await task_a() 时，main 就“停”在了这一行，手里的执行权交给了 task_a。
  - 此时，task_b 甚至还没有被调用，它连队都没排上。
  - 只有等 task_a 睡完 2 秒并打印了 A end 之后，main 才会醒来往下走，去执行 await task_b()。
- 结论：在同一个协程里连续写 await，效果和普通的同步代码（串行）是一模一样的。

In [ ]:
import asyncio
import time

async def task_a():
    print("A start")
    await asyncio.sleep(2)
    print("A end")
    return "A"

async def task_b():
    print("B start")
    await asyncio.sleep(1)
    print("B end")
    return "B"

async def main():
    start = time.time()
    
    # 注意这里的调用方式
    res_a = await task_a()
    res_b = await task_b()
    
    print(f"Results: {res_a}, {res_b}")
    print(f"Total time: {time.time() - start:.2f}s")

if __name__ == "__main__":
    asyncio.run(main())

### 2.2 gather 的并发逻辑
Q: 现在我们将代码稍作修改，问题如下：

- 此时的打印顺序是怎样的？
- 总耗时接近几秒？
- results 列表里的顺序是按“完成时间”排序，还是按“传入顺序”排序？

A：打印顺序是A start，B start，B end，A end，总耗时2秒，results顺序按照传入顺序输出。

In [ ]:
async def main():
    start = time.time()
    
    # 使用 gather
    results = await asyncio.gather(task_a(), task_b())
    
    print(f"Results: {results}")
    print(f"Total time: {time.time() - start:.2f}s")

### 2.3 create_task 的执行时机
Q: 这道题考察你对“调度权”的理解。请仔细看：
- Task 1 begins 会在 Main is busy... 之前打印，还是在 Main is finally ready... 之后打印？
- 为什么？（提示：create_task 只是“提交”了任务，什么时候“开始跑”需要看谁手里有球）。

A: 在Main is finally ready... 之后打印：
- 调度员（Event Loop）只有一个：在你的 main 函数运行期间，它手里紧紧攥着“执行权”这颗球。
- create_task 只是“报名”：当你调用它时，simple_task 只是去事件循环那里登记了一下，排在了队尾。
- time.sleep 是“霸凌”：它直接让整个线程（包括调度员）原地休克 1 秒。在这 1 秒里，调度员没法看队列，所以 simple_task 只能乖乖排队。
- 只有 await 是“让球”：直到代码运行到 await t1 时，main 终于说：“我等累了，球给调度员吧。” 此时调度员才回过头看队列，发现了 simple_task 并开始执行。

In [ ]:
import asyncio
import time

async def simple_task(id, delay):
    print(f"Task {id} begins")
    await asyncio.sleep(delay)
    print(f"Task {id} finishes")

async def main():
    print("Main starts")
    
    # 步骤 1: 创建任务（放入事件循环队列）
    t1 = asyncio.create_task(simple_task(1, 3))
    
    # 步骤 2: 模拟一段同步 CPU 计算（比如处理一个很大的本地循环）
    print("Main is busy...")
    time.sleep(1)  # <--- 注意：这是 time.sleep，同步阻塞
    print("Main is finally ready to await")
    
    # 步骤 3: 真正的异步等待
    await t1
    print("Main ends")

if __name__ == "__main__":
    asyncio.run(main())

### 2.4 那个致命的“毒药”
Q：Async job started 和 Blocking job started 会同时出现吗？整个程序的行为会变成怎样？

A：Async job started 和 Blocking job started 会同时出现，然后先输出Blocking job finished，最后输出Async job finished整体耗时5秒

In [ ]:
async def async_job():
    print("Async job started")
    await asyncio.sleep(2)
    print("Async job finished")

async def blocking_job():
    print("Blocking job started")
    time.sleep(5)  # <--- 注意这里！
    print("Blocking job finished")

async def main():
    start = time.time()
    
    # 试图并发运行一个异步任务和一个阻塞任务
    await asyncio.gather(async_job(), blocking_job())
    
    print(f"Total time: {time.time() - start:.2f}s")

### 2.5 异常传播
Q：这是关于 gather 容错处理的最后一道大关。请问：
- 当 1 秒后 error_task 抛出异常时，main 会立刻进入 except 分支吗？
- 此时，那个需要跑 2 秒的 normal_task 会发生什么？它会：
  - A: 继续在后台跑完，并在第 2 秒打印 "I finally finished!"。
  - B: 被 gather 自动取消，打印 "I was cancelled!"。
  - C: 继续跑，但是它的结果会被丢弃，且不会打印任何东西。

A：gather对于tasks中的任意task报错时，会立刻停止等待并把错误甩给调用者，但它并不会去帮把剩下的任务“掐死”。
- 立刻进入 except 吗？ 是的。在第 1 秒，error_task 报错时，await asyncio.gather 会立刻感受到这个异常，并把它抛给 main。
- normal_task 会发生什么？ 选 A。它会继续在后台跑完，并在第 2 秒打印 "I finally finished!"。

In [ ]:
import asyncio

async def error_task():
    await asyncio.sleep(1)
    print("Error task: raising exception now!")
    raise ValueError("Oops!")

async def normal_task():
    try:
        await asyncio.sleep(2)
        print("Normal task: I finally finished!")
        return "Success"
    except asyncio.CancelledError:
        print("Normal task: I was cancelled!")
        raise

async def main():
    try:
        # 同时跑这两个任务
        res = await asyncio.gather(error_task(), normal_task())
        print(f"Results: {res}")
    except Exception as e:
        print(f"Main caught error: {e}")
    
    # 重点：这里等一会儿，看看 normal_task 还在不在跑
    await asyncio.sleep(2)
    print("Main exit")

if __name__ == "__main__":
    asyncio.run(main())

### 优化方案
- 做法 1：手动取消（旧时代做法），在 main 的 except 块里手动处理剩下的任务。
- 做法 2：使用 asyncio.TaskGroup (Python 3.11+ 推荐)，这是现代异步编程的标准方案。

In [ ]:
async def main():
    try:
        async with asyncio.TaskGroup() as tg:
            tg.create_task(error_task())
            tg.create_task(normal_task())
    except ExceptionGroup as eg:
        print(f"Caught: {eg}")